In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-uj8s3fmy/unsloth_b767477e5e084ef388401e97a9c23696
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-uj8s3fmy/unsloth_b767477e5e084ef388401e97a9c23696
  Resolved https://github.com/unslothai/unsloth.git to commit aa5832de9282987ae6221dfac1877d23d64cad9a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from datasets import load_dataset

# Load your dataset from the uploaded 'train.jsonl' file.
# We specify 'json' as the format and provide the file path.
dataset = load_dataset("json", data_files="train.jsonl", split="train")

# Let's look at a sample from the dataset to see what it looks like.
print("Original dataset sample:")
print(dataset[0])

Original dataset sample:
{'Instruction': 'What is the difference between a petition and a plaint in Indian law?', 'Response': "A petition is a formal request submitted to a courttribunalor authority to seek a specific remedy or relief. It is commonly used for various purposessuch as filing a writ petition in the High Court or submitting a petition for divorce. On the other handa plaint is a formal written statement of a plaintiff's claim in a civil lawsuit. The key difference is that a petition is more versatile and can be used for various legal matterswhile a plaint is specific to civil cases."}


In [3]:
# Step 3: Load the Pre-trained Model and Tokenizer
# ------------------------------------------------
# We will now load the Qwen3-4B model. We're using a special 4-bit quantized version
# from Unsloth's model zoo, which is highly optimized for memory and speed.

from unsloth import FastLanguageModel
import torch

# This is the name of the new model we are using.
model_name = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"

# Let's load the model.
# max_seq_length: This is the maximum number of tokens the model can handle in one go.
# dtype: We leave this as None to let Unsloth decide the best data type.
# load_in_4bit: This is the key to memory saving! It loads the model in 4-bit precision.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Now, we need to format our dataset into the specific chat template that the Qwen3 model expects.
# The template structures the conversation with roles like 'user' and 'assistant'.
# This function is the same as the one for Qwen2, as they share a similar chat structure.

def formatting_prompts_func(examples):
    instructions = examples["Instruction"]
    responses = examples["Response"]
    texts = []
    for instruction, response in zip(instructions, responses):
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

# Let's see how the formatted prompt looks.
print("\nFormatted dataset sample:")
print(formatted_dataset[0]['text'])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/24607 [00:00<?, ? examples/s]


Formatted dataset sample:
<|im_start|>user
What is the difference between a petition and a plaint in Indian law?<|im_end|>
<|im_start|>assistant
<think>

</think>

A petition is a formal request submitted to a courttribunalor authority to seek a specific remedy or relief. It is commonly used for various purposessuch as filing a writ petition in the High Court or submitting a petition for divorce. On the other handa plaint is a formal written statement of a plaintiff's claim in a civil lawsuit. The key difference is that a petition is more versatile and can be used for various legal matterswhile a plaint is specific to civil cases.<|im_end|>



In [4]:
# Step 4: Configure LoRA for Efficient Fine-Tuning
# -------------------------------------------------
# Instead of training the entire model, we use LoRA (Low-Rank Adaptation).
# It freezes the main model and only trains small, additional "adapter" layers. This is much more efficient.

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
    max_seq_length=2048,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.1 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:

# --------------------------
# The `SFTTrainer` (Supervised Fine-Tuning Trainer) will handle the training loop for us.

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=100, # You can increase this for better performance, e.g., to 200 or more.
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none", # Disables wandb logging
    ),
)



Map (num_proc=2):   0%|          | 0/24607 [00:00<?, ? examples/s]

In [6]:
# Step 6: Start Fine-Tuning!
# --------------------------
# This cell will start the actual training process.

print("\nStarting the fine-tuning process...")
trainer.train()
print("Fine-tuning completed successfully!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Starting the fine-tuning process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,607 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss
1,3.526800
2,2.836400
3,3.660200
4,2.908600
5,2.767500
6,2.508600
7,2.686100
8,2.034900
9,2.130100
10,1.975500


Unsloth: Will smartly offload gradients to save VRAM!
Fine-tuning completed successfully!


In [7]:
# ----------------------------------------
# Step 7: Inference (Chat with Your Model)
# ----------------------------------------

import re
from unsloth import FastLanguageModel

# Prepare the model for inference
FastLanguageModel.for_inference(model)

# Get user question
user_question = input("Please enter your legal question: ")

# Format the prompt for the model
messages = [
    {"role": "user", "content": user_question},
]

# Tokenize with chat template
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# Generate the response
outputs = model.generate(
    input_ids=input_ids,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.2,      # Lower for factual answers
    top_p=0.9,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id
)

# Decode only newly generated tokens
response = tokenizer.decode(
    outputs[0][len(input_ids[0]):],
    skip_special_tokens=True
)

# -----------------------------
# Remove HTML-like tags from output
# -----------------------------
clean_response = re.sub(r"<.*?>", "", response).strip()

# Print the final result
print("\n--- Model Response ---")
print("Your Question:", user_question)
print("Answer:", clean_response)


Please enter your legal question: What is the difference between a petition and a plaint in Indian law?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Model Response ---
Your Question: What is the difference between a petition and a plaint in Indian law?
Answer: In Indian law, a petition refers to a document filed by a party seeking a remedy or relief from a court. A plaint, on the other hand, is a specific type of petition that is filed in a civil court to initiate a civil suit. It contains the details of the case, including the nature of the dispute, the relief sought, and the facts of the case. The main difference between a petition and a plaint is that a petition is a general term for any document filed with a court, while a plaint is a specific type of petition used in civil cases.


In [8]:

# Step 9: Model Evaluation for Research Paper
# -------------------------------------------
# This step calculates standard NLP metrics (ROUGE and BERTScore) to evaluate
# the model's performance on a test set. This is crucial for a research paper.

# First, install the necessary evaluation libraries.
!pip install evaluate rouge_score bert_score

import evaluate
import pandas as pd
from tqdm import tqdm
import re

# Load the ROUGE and BERTScore metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load("bertscore") # Corrected from "bert-score"

# Create a small, random evaluation set from your original dataset.
# We'll use 100 samples for a quick but effective evaluation.
eval_dataset = dataset.shuffle(seed=42).select(range(100))

# Prepare the model for inference one more time to be safe.
FastLanguageModel.for_inference(model)

# We will store the model's predictions and the original answers (references)
predictions = []
references = []

print("\nRunning evaluation on 100 samples...")

# Loop through the evaluation dataset
for item in tqdm(eval_dataset):
    # Get the instruction (question)
    instruction = item["Instruction"]
    # Get the reference (ground-truth answer)
    reference = item["Response"]

    # Format the prompt for the model
    messages = [
        {"role": "user", "content": instruction},
    ]

    # Tokenize the input
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate the model's response
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the response and clean it
    prediction = tokenizer.decode(outputs[0][len(input_ids[0]):], skip_special_tokens=True)
    clean_prediction = re.sub(r"<.*?>", "", prediction).strip()

    # Add the results to our lists
    predictions.append(clean_prediction)
    references.append(reference)

# Now, calculate the scores
print("\nCalculating evaluation metrics...")
rouge_results = rouge.compute(predictions=predictions, references=references)
bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")

# Format and print the results in a clean table for your paper
results_df = pd.DataFrame({
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1'],
    'Score': [
        rouge_results['rouge1'],
        rouge_results['rouge2'],
        rouge_results['rougeL'],
        sum(bert_results['precision']) / len(bert_results['precision']),
        sum(bert_results['recall']) / len(bert_results['recall']),
        sum(bert_results['f1']) / len(bert_results['f1'])
    ]
})

print("\n--- Evaluation Results ---")
print(results_df.to_string(index=False))

# You can copy this table directly into your research paper.
# ROUGE scores measure n-gram overlap.
# BERTScore measures semantic similarity, which is often a better indicator of quality.


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5fd43138ae3e6a90df33686334ed2678d56a8324acc209be159dc0ebfbc18681
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score



Running evaluation on 100 samples...


100%|██████████| 100/100 [13:18<00:00,  7.98s/it]



Calculating evaluation metrics...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Evaluation Results ---
      Metric    Score
     ROUGE-1 0.447500
     ROUGE-2 0.274903
     ROUGE-L 0.366758
 BERTScore-P 0.895151
 BERTScore-R 0.883556
BERTScore-F1 0.888830


In [16]:
# ------------------------------------------------------------
# One-cell: Save & verify Unsloth Qwen3-4B fine-tuned model
# ------------------------------------------------------------
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import os

# Your fine-tuned model & tokenizer (already loaded or from pretrained)
model_name = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"

# Load the model and tokenizer (skip if already loaded)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    load_in_4bit=True,  # keep memory-efficient quantization
)

# Directory to save fine-tuned weights
save_dir = "qwen3_4b_legal_finetuned"
os.makedirs(save_dir, exist_ok=True)

# Save model & tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Verify files saved
print(f"✅ Model and tokenizer saved successfully in '{save_dir}/'")
print("Saved files:", os.listdir(save_dir))

# Reload to confirm
model_reloaded, tokenizer_reloaded = FastLanguageModel.from_pretrained(
    model_name=save_dir,
    load_in_4bit=True,
)

print("✅ Reload test passed — model & tokenizer loaded again successfully.")



==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

✅ Model and tokenizer saved successfully in 'qwen3_4b_legal_finetuned/'
Saved files: ['tokenizer_config.json', 'vocab.json', 'chat_template.jinja', 'generation_config.json', 'special_tokens_map.json', 'added_tokens.json', 'tokenizer.json', 'config.json', 'merges.txt', 'model.safetensors']
==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Reload test passed — model & tokenizer loaded again successfully.


In [17]:
from google.colab import files
files.download("qwen3_4b_legal_finetuned.zip")

FileNotFoundError: Cannot find file: qwen3_4b_legal_finetuned.zip

In [1]:
# Step 11: Save Merged Model and Tokenizer
# ------------------------------------------
# This step loads the fine-tuned LoRA adapters and merges them into the base model,
# saving the result as a complete, standalone model for use with Hugging Face transformers.

from unsloth import FastLanguageModel

# Reload the model from the saved LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen3_4b_legal_finetuned_lora",
)

# Merge and save the model
merged_model_dir = "qwen3_4b_legal_finetuned_merged"
model.save_pretrained_merged(merged_model_dir, tokenizer)

print(f"\nFull merged model saved to '{merged_model_dir}'.")
print("You can now download this folder for local use with the Hugging Face transformers library.")



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.10.1 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [02:47<02:47, 167.87s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:53<00:00, 56.81s/it]


Unsloth: Merge process complete. Saved to `/content/qwen3_4b_legal_finetuned_merged`

Full merged model saved to 'qwen3_4b_legal_finetuned_merged'.
You can now download this folder for local use with the Hugging Face transformers library.


In [3]:
# Step 11: Prepare and Run Model with Ollama (Offline)
# ----------------------------------------------------
# This step converts your fine-tuned model to GGUF format, which is what Ollama uses.
# It then guides you on how to set it up for local, offline use.

# Cell 11.1: Compile llama.cpp for GGUF Conversion
# This is a necessary step to ensure the quantization tool works correctly in the Colab environment.
print("\nCompiling llama.cpp for GGUF conversion...")
!git clone --recursive https://github.com/ggerganov/llama.cpp
!cd llama.cpp && make clean && make all -j
print("llama.cpp compiled successfully.")

# Cell 11.2: Convert Merged Model to GGUF
# This step loads the fully merged model you created in Step 10 and then
# converts it into the GGUF format.
from unsloth import FastLanguageModel

print("\nReloading the merged model for GGUF conversion...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen3_4b_legal_finetuned_merged",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Save the loaded model as a GGUF file.
# We use 'q8_0' quantization for a good balance of performance and file size.
print("\nSaving model in GGUF format...")
model.save_pretrained_gguf("qwen3_4b_legal_finetuned.gguf", tokenizer, quantization_method="q8_0")
print("\nModel successfully converted to 'qwen3_4b_legal_finetuned.gguf'.")


Compiling llama.cpp for GGUF conversion...
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
llama.cpp compiled successfully.

Reloading the merged model for GGUF conversion...
==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]


Saving model in GGUF format...
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 4.42 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 36/36 [00:01<00:00, 31.54it/s]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving qwen3_4b_legal_finetuned.gguf/pytorch_model.bin...
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at qwen3_4b_legal_finetuned.gguf into q8_0 GGUF format.
The output location will be /content/qwen3_4b_legal_finetuned.gguf/unsloth.Q8_0.gguf
This might take 3 minutes...
  File "/content/llama.cpp/convert_hf_to_gguf.py", line 34
    try:
    ^^^
IndentationError: expected an indented block after 'try' statement on line 33


RuntimeError: Unsloth: Quantization failed for /content/qwen3_4b_legal_finetuned.gguf/unsloth.Q8_0.gguf
You might have to compile llama.cpp yourself, then run this again.
You do not need to close this Python program. Run the following commands in a new terminal:
You must run this in the same folder as you're saving your model.
git clone --recursive https://github.com/ggerganov/llama.cpp
cd llama.cpp && make clean && make all -j
Once that's done, redo the quantization.